# Flickr30k Final Artifact Audit

Kiểm toán cuối chuỗi kết quả đã khóa. Notebook không đọc dữ liệu ảnh,
caption Test hay checkpoint; không huấn luyện và không đánh giá lại.

## Input bắt buộc — đúng 4 output

1. `F30K_Global_Val_Lock`;
2. `F30K_T3S3_HeldOut_Test_OneTime`;
3. `F30K_Paper_Tables`;
4. `F30K_Final_Closeout`.

CPU, Accelerator None, Internet Off; dự kiến dưới 1 phút.


## Cell 1 — Kiểm toán manifest, khóa Test và tính nhất quán


In [1]:
import hashlib
import json
import os
import shutil
import zipfile
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
EXPERIMENT_ID = "F30K_Final_Audit"
WORK_ROOT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT", f"/kaggle/working/{EXPERIMENT_ID}"
))
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def now_utc():
    return datetime.now(timezone.utc).isoformat()

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def unique_named(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (f"Cần đúng một {name}", matches)
    return matches[0]

def load_json(name):
    path = unique_named(name)
    return path, json.loads(path.read_text(encoding="utf-8"))

def write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

def verify_manifest(root):
    manifest_path = root / "MANIFEST.json"
    assert manifest_path.is_file(), manifest_path
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    rows = manifest["artifacts"]
    for row in rows:
        path = root / row["relative_path"]
        assert path.is_file(), path
        assert path.stat().st_size == int(row["size_bytes"]), path
        assert sha256(path) == row["sha256"], path
    return manifest_path, manifest


global_path, global_lock = load_json("F30K_GLOBAL_VALIDATION_LOCK.json")
heldout_path, heldout = load_json(
    "F30K_T3S3_HeldOut_Test_OneTime_RESULT.json"
)
tables_path, paper_tables = load_json("F30K_Paper_Tables_RESULT.json")
closeout_path, closeout = load_json("F30K_Final_Closeout_RESULT.json")

verified = {}
for name, result_path in [
    ("heldout", heldout_path),
    ("paper_tables", tables_path),
    ("final_closeout", closeout_path),
]:
    manifest_path, manifest = verify_manifest(result_path.parent)
    verified[name] = {
        "manifest_sha256": sha256(manifest_path),
        "artifact_count": len(manifest["artifacts"]),
    }

assert global_lock["status"] == "locked"
assert global_lock["selected_method"]["pair_id"] == "T3S3_LogitKD"
assert heldout["status"] == "complete_locked_no_further_test_access"
assert heldout["held_out_test_authorization_consumed"] is True
assert heldout["held_out_test_authorization_remaining"] == 0
assert heldout["test_set_status"] == "opened_once_then_locked"
assert paper_tables["status"] == "complete"
assert paper_tables["held_out_test_reopened"] is False
assert closeout["status"] == "scientific_evaluation_complete"
assert closeout["scientific_boundaries"][
    "held_out_test_authorization_remaining"
] == 0
assert closeout["selected_method"]["checkpoint_sha256"] == heldout[
    "checkpoint_sha256"
]
assert closeout["held_out_test_metrics_x100"] == heldout[
    "metrics_x100_rounded_2dp"
]

audit = {
    "schema_version": "f30k_final_artifact_audit_v1",
    "created_at_utc": now_utc(),
    "status": "passed",
    "experiment_id": EXPERIMENT_ID,
    "verified_manifests": verified,
    "selected_pair": closeout["selected_method"]["pair_id"],
    "selected_method": closeout["selected_method"]["method"],
    "checkpoint_sha256": closeout["selected_method"]["checkpoint_sha256"],
    "held_out_metrics_x100": closeout["held_out_test_metrics_x100"],
    "held_out_test_reopened": False,
    "held_out_test_authorization_remaining": 0,
    "scientific_evaluation_complete": True,
    "kaggle_compute_complete": True,
    "remaining_external_work": [
        "Android on-device latency, memory and thermal benchmark",
        "write scientific report from locked artifacts",
    ],
}
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
write_json(result_path, audit)

report_path = WORK_ROOT / "F30K_FINAL_AUDIT.md"
report_path.write_text(
    f"""# Flickr30k — kiểm toán cuối

- Trạng thái: **PASSED**.
- Phương pháp: **{audit['selected_pair']} / {audit['selected_method']}**.
- Checkpoint SHA-256: `{audit['checkpoint_sha256']}`.
- Held-out Test còn quyền chạy: **0**.
- CIDEr-D Held-out Test: **{audit['held_out_metrics_x100']['CIDEr_D']:.2f}**.
- Tính toán Kaggle: **hoàn tất**.
- Công việc ngoài Kaggle: benchmark Android thật và biên soạn báo cáo.
""",
    encoding="utf-8",
)

manifest_path = WORK_ROOT / "MANIFEST.json"
artifacts = [result_path, report_path]
write_json(manifest_path, {
    "schema_version": "f30k_final_audit_manifest_v1",
    "created_at_utc": now_utc(),
    "artifact_count": len(artifacts),
    "artifacts": [
        {
            "relative_path": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for path in artifacts
    ],
})
verify_manifest(WORK_ROOT)

print("Final audit            : PASSED")
print("Verified manifests     :", verified)
print("Held-out Test remaining: 0")
print("Kaggle compute         : COMPLETE")
print("Next                    : REPORT + ANDROID BENCHMARK")
print("Result                  :", result_path)
print("Manifest                :", manifest_path)


Final audit            : PASSED
Verified manifests     : {'heldout': {'manifest_sha256': 'e6d7503464dd5fd8706d92ac3e5b1b1eb0a20753107e96d5930c6f55f53b8ec8', 'artifact_count': 4}, 'paper_tables': {'manifest_sha256': 'e4e7c3b06589cc7524dad82d89dbe49f902c07a9d504e653522657b929dc306c', 'artifact_count': 5}, 'final_closeout': {'manifest_sha256': '7283bad52019879c16f5b8b9afbea21de4f43bb9674cea552157f2ac0305fbf7', 'artifact_count': 2}}
Held-out Test remaining: 0
Kaggle compute         : COMPLETE
Next                    : REPORT + ANDROID BENCHMARK
Result                  : /kaggle/working/F30K_Final_Audit/F30K_Final_Audit_RESULT.json
Manifest                : /kaggle/working/F30K_Final_Audit/MANIFEST.json
